In [137]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [138]:
data_path = "dataset/Electronic_sales_Sep2023-Sep2024.csv"
df = pd.read_csv(data_path)

print(df.info())
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Customer ID        20000 non-null  int64  
 1   Age                20000 non-null  int64  
 2   Gender             19999 non-null  str    
 3   Loyalty Member     20000 non-null  str    
 4   Product Type       20000 non-null  str    
 5   SKU                20000 non-null  str    
 6   Rating             20000 non-null  int64  
 7   Order Status       20000 non-null  str    
 8   Payment Method     20000 non-null  str    
 9   Total Price        20000 non-null  float64
 10  Unit Price         20000 non-null  float64
 11  Quantity           20000 non-null  int64  
 12  Purchase Date      20000 non-null  str    
 13  Shipping Type      20000 non-null  str    
 14  Add-ons Purchased  15132 non-null  str    
 15  Add-on Total       20000 non-null  float64
dtypes: float64(3), int64(4), str(9)
m

,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
0,1000,53,Male,No,Smartphone,SKU1004,2,Cancelled,Credit Card,5538.33,791.19,7,2024-03-20,Standard,"Accessory,Accessory,Accessory",40.21
1,1000,53,Male,No,Tablet,SKU1002,3,Completed,Paypal,741.09,247.03,3,2024-04-20,Overnight,Impulse Item,26.09
2,1002,41,Male,No,Laptop,SKU1005,3,Completed,Credit Card,1855.84,463.96,4,2023-10-17,Express,NaN,0.00
3,1002,41,Male,Yes,Smartphone,SKU1004,2,Completed,Cash,3164.76,791.19,4,2024-08-09,Overnight,"Impulse Item,Impulse Item",60.16
4,1003,75,Male,Yes,Smartphone,SKU1001,5,Completed,Cash,41.50,20.75,2,2024-05-21,Express,Accessory,35.56


In [139]:
# Note: Add-ons dihilangkan dulu untuk sementara (nanti coba aja lagi aowkwk)

df = df[df['Order Status'] == 'Completed']
df = df.drop(columns=['Customer ID', 'Age', 'Gender', 'Loyalty Member', 'Rating', 'Payment Method','Order Status', 'Add-ons Purchased', 'Add-on Total', 'Shipping Type'])

df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])
df['YearMonth'] = df['Purchase Date'].dt.to_period('M')

df.head()

,Product Type,SKU,Total Price,Unit Price,Quantity,Purchase Date,YearMonth
1,Tablet,SKU1002,741.09,247.03,3,2024-04-20,2024-04
2,Laptop,SKU1005,1855.84,463.96,4,2023-10-17,2023-10
3,Smartphone,SKU1004,3164.76,791.19,4,2024-08-09,2024-08
4,Smartphone,SKU1001,41.50,20.75,2,2024-05-21,2024-05
5,Smartphone,SKU1001,83.00,20.75,4,2024-05-26,2024-05


In [140]:
print("Null Value")
print(df.isnull().sum())

print("\nDuplicated Value")
print(df.duplicated().sum())

Null Value
Product Type     0
SKU              0
Total Price      0
Unit Price       0
Quantity         0
Purchase Date    0
YearMonth        0
dtype: int64

Duplicated Value
2561


In [141]:
df = df.groupby(['YearMonth', 'Product Type', 'SKU']).agg(
    Total_Quantity=('Quantity', 'sum'),
    Total_Revenue=('Total Price', 'sum'),
    Average_Unit_Price=('Unit Price', 'mean')
).reset_index()

df = df.sort_values(by=['SKU', 'YearMonth']).reset_index(drop=True)

print(df.shape[0])
df.head(10)

113


,YearMonth,Product Type,SKU,Total_Quantity,Total_Revenue,Average_Unit_Price
0,2024-01,Headphones,HDP456,1013,365875.34,361.18
1,2024-02,Headphones,HDP456,704,254270.72,361.18
2,2024-03,Headphones,HDP456,895,323256.10,361.18
3,2024-04,Headphones,HDP456,860,310614.80,361.18
4,2024-05,Headphones,HDP456,783,282803.94,361.18
5,2024-06,Headphones,HDP456,958,346010.44,361.18
6,2024-07,Headphones,HDP456,833,300862.94,361.18
7,2024-08,Headphones,HDP456,887,320366.66,361.18
8,2024-09,Headphones,HDP456,627,226459.86,361.18
9,2024-01,Laptop,LTP123,936,631163.52,674.32


In [142]:
df['Year'] = df['YearMonth'].dt.year
df['Month'] = df['YearMonth'].dt.month

df_original = df.copy()

df = pd.get_dummies(df, columns=['Product Type', 'SKU'], drop_first=True)

print(df.shape[0])
df.head()

113


,YearMonth,Total_Quantity,Total_Revenue,Average_Unit_Price,Year,Month,Product Type_Laptop,Product Type_Smartphone,Product Type_Smartwatch,Product Type_Tablet,SKU_LTP123,SKU_SKU1001,SKU_SKU1002,SKU_SKU1003,SKU_SKU1004,SKU_SKU1005,SKU_SMP234,SKU_SWT567,SKU_TBL345
0,2024-01,1013,365875.34,361.18,2024,1,False,False,False,False,False,False,False,False,False,False,False,False,False
1,2024-02,704,254270.72,361.18,2024,2,False,False,False,False,False,False,False,False,False,False,False,False,False
2,2024-03,895,323256.10,361.18,2024,3,False,False,False,False,False,False,False,False,False,False,False,False,False
3,2024-04,860,310614.80,361.18,2024,4,False,False,False,False,False,False,False,False,False,False,False,False,False
4,2024-05,783,282803.94,361.18,2024,5,False,False,False,False,False,False,False,False,False,False,False,False,False


In [143]:
unique_month = sorted(df['YearMonth'].unique())
test_time_limit = unique_month[-2]

train_data = df[df['YearMonth'] < test_time_limit]
test_data = df[df['YearMonth'] >= test_time_limit]

y_train = train_data['Total_Quantity']
X_train = train_data.drop(columns=['YearMonth', 'Total_Quantity', 'Total_Revenue'])

y_test = test_data['Total_Quantity']
X_test = test_data.drop(columns=['YearMonth', 'Total_Quantity', 'Total_Revenue'])

model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

prediction_test = model.predict(X_test)
error = mean_absolute_error(y_test, prediction_test)

print(f"Rata-rata tebakan model meleset sebanyak: {error:.2f} unit produk")

print("\n=== INFO JUMLAH BARIS ===")
print(f"Data Training (Masa Lalu): {X_train.shape[0]} baris")
print(f"Data Testing (Soal Ujian): {X_test.shape[0]} baris")
print(f"Total Baris Keseluruhan: {df.shape[0]} baris")

rata_rata_asli = y_test.mean()
print(f"Rata-rata penjualan asli per bulan: {rata_rata_asli:.2f} unit")
print(f"Persentase Error: {(error / rata_rata_asli) * 100:.2f}%")

Rata-rata tebakan model meleset sebanyak: 90.98 unit produk

=== INFO JUMLAH BARIS ===
Data Training (Masa Lalu): 93 baris
Data Testing (Soal Ujian): 20 baris
Total Baris Keseluruhan: 113 baris
Rata-rata penjualan asli per bulan: 643.75 unit
Persentase Error: 14.13%


In [ ]:
# Testing only (please remove after this test)

import pulp
import numpy as np

# 1. BIKIN DICTIONARY DARI HASIL PREDIKSI ML
hasil_prediksi = df_original.loc[test_data.index, ['SKU']].copy()
hasil_prediksi['Tebakan_Demand'] = np.round(prediction_test).astype(int)

# Bikin agregasi biar per SKU cuma ada 1 tebakan demand tertinggi
demand_bulan_depan = hasil_prediksi.groupby('SKU')['Tebakan_Demand'].max().to_dict()

# Kita ambil rata-rata harga satuan per SKU dari data historis
harga_satuan = df_original.groupby('SKU')['Average_Unit_Price'].mean().to_dict()

# 2. SETUP VARIABEL BISNIS (Silakan diubah sesuai skenario dosenmu)
MODAL_RESTOCK = 1000000     # Misal budget maksimal $500,000
KAPASITAS_GUDANG = 10000    # Gudang cuma muat 2000 unit barang
MARGIN_PROFIT = 0.3        # Asumsi untung 30% dari harga beli

# 3. INISIALISASI MODEL PuLP
model = pulp.LpProblem("Optimasi_Restock_Toko_Elektronik", pulp.LpMaximize)

# 4. BIKIN VARIABEL KEPUTUSAN (Berapa unit yang mau dibeli per SKU?)
jumlah_beli = {
    sku: pulp.LpVariable(f"Beli_{sku}", lowBound=10, cat='Integer') 
    for sku in demand_bulan_depan.keys()
}

# 5. OBJECTIVE FUNCTION: Maksimalkan Profit!
model += pulp.lpSum([jumlah_beli[sku] * harga_satuan[sku] * MARGIN_PROFIT for sku in demand_bulan_depan.keys()]), "Total_Profit"

# 6. CONSTRAINTS (Aturan Main)
model += pulp.lpSum([jumlah_beli[sku] * harga_satuan[sku] for sku in demand_bulan_depan.keys()]) <= MODAL_RESTOCK, "Batas_Modal"

model += pulp.lpSum([jumlah_beli[sku] for sku in demand_bulan_depan.keys()]) <= KAPASITAS_GUDANG, "Batas_Gudang"

for sku in demand_bulan_depan.keys():
    model += jumlah_beli[sku] <= demand_bulan_depan[sku], f"Batas_Demand_{sku}"

# 7. EXECUTE!
model.solve()

# 8. TAMPILKAN HASILNYA
print(f"Model Status: {pulp.LpStatus[model.status]}")
print(f"Base Budget: ${MODAL_RESTOCK:,.2f}")
print(f"Inventory Capacity: {KAPASITAS_GUDANG} unit\n")

print("Next Month's Restock Recommendations\n")
total_biaya = 0
total_unit = 0

for sku in demand_bulan_depan.keys():
    unit_dibeli = int(jumlah_beli[sku].varValue)
    biaya = unit_dibeli * harga_satuan[sku]
    
    total_biaya += biaya
    total_unit += unit_dibeli
    
    print(f"{sku}: Buy {unit_dibeli} unit (Demand Prediction: {demand_bulan_depan[sku]} unit)")

print(f"\nTotal budget used: ${total_biaya:,.2f}")
print(f"Budget Leftovers: ${MODAL_RESTOCK - total_biaya:,.2f}")
print(f"Total Unit in Inventory: {total_unit} unit")
print(f"Estimated Profit: ${pulp.value(model.objective):,.2f}")

Model Status: Optimal
Base Budget: $1,000,000.00
Inventory Capacity: 10000 unit

Next Month's Restock Recommendations

HDP456: Buy 303 unit (Demand Prediction: 863 unit)
LTP123: Buy 10 unit (Demand Prediction: 851 unit)
SKU1001: Buy 592 unit (Demand Prediction: 597 unit)
SKU1002: Buy 603 unit (Demand Prediction: 603 unit)
SKU1003: Buy 364 unit (Demand Prediction: 649 unit)
SKU1004: Buy 10 unit (Demand Prediction: 646 unit)
SKU1005: Buy 587 unit (Demand Prediction: 588 unit)
SMP234: Buy 10 unit (Demand Prediction: 811 unit)
SWT567: Buy 216 unit (Demand Prediction: 884 unit)
TBL345: Buy 10 unit (Demand Prediction: 951 unit)

Total budget used: $1,000,000.00
Budget Leftovers: $0.00
Total Unit in Inventory: 2705 unit
Estimated Profit: $300,000.00
